# **Part 1: Input Area**

In [ ]:
#@title #**Enter Keywords to Search** (Separate multiple keywords with commas; do not use full product names)
PRODUCT = "Microsoft SharePoint" #@param {type:"string"}

In [ ]:
# @title Authentication, Drive Mounting, and Configuration
# ===========================
# AUTHENTICATION
# ===========================
import gspread
from google.colab import auth, drive
from googleapiclient.discovery import build
import google.auth
# Authenticate the user for API access (required for gspread and drive_service)
auth.authenticate_user()
creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
gc = gspread.authorize(creds)
drive_service = build('drive', 'v3', credentials=creds)
print("🔑 Google user authenticated.")

# ===========================
# CONFIG
# ===========================
# shared drive folder ID (Ensure this is correct)
PARENT_FOLDER_ID = "1Z8m_0t4xeyxlz2oJug3aCMbj4z2-Xg9W"

# ===========================
# MOUNT DRIVE
# ===========================
# Mount Google Drive to /content/drive
drive.mount('/content/drive', force_remount=True)

print(f"✅ Environment ready. Drive mounted and authenticated. Parent Folder ID set to: {PARENT_FOLDER_ID}")

🔑 Google user authenticated.
Mounted at /content/drive
✅ Environment ready. Drive mounted and authenticated. Parent Folder ID set to: 1Z8m_0t4xeyxlz2oJug3aCMbj4z2-Xg9W


In [ ]:
#@title Token
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io, json, os

# Authenticate to Google
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Your file ID from Google Drive
FILE_ID = "1MW4L2WgpoZ30Jqzdqlg8a6c8_Xwnm809"

# Download the JSON config
request = drive_service.files().get_media(fileId=FILE_ID)
fh = io.BytesIO()
downloader = MediaIoBaseDownload(fh, request)
done = False
while not done:
    status, done = downloader.next_chunk()

fh.seek(0)
cfg = json.load(fh)

# Set both environment variables
os.environ["OPENAI_API_KEY"] = cfg.get("OPENAI_API_KEY", "")
os.environ["API_KEY"] = cfg.get("API_KEY", "")

print("OPENAI_API_KEY loaded:", bool(os.environ["OPENAI_API_KEY"]))
print("API_KEY loaded:", bool(os.environ["API_KEY"]))

OPENAI_API_KEY loaded: True
API_KEY loaded: True


In [ ]:
#@title Input Dataframe
# Import libraries
import pandas as pd
import gspread
from google.colab import auth
import google.auth

# Authenticate with Google
auth.authenticate_user()
creds, _ = google.auth.default()
gc = gspread.authorize(creds)

# Open Google Sheet by URL
spreadsheet = gc.open_by_url("https://docs.google.com/spreadsheets/d/1dC25JdU4e9Ww1IhFr1vcZOgsuzFzXvdLDA251vfLKdE/edit?gid=0#gid=0")

# Select the first worksheet (or a specific one)
worksheet = spreadsheet.worksheet("Portfolio")

# Get all values (evaluated, not formulas)
data = worksheet.get_all_values()

# Get all values
data = worksheet.get_all_values()

# Set row 3 as header
header = data[0]

# Load data starting from row 4
df_input = pd.DataFrame(data[1:], columns=header)
input_ports = df_input['id']
input_ports


,id
0,9a779437-dc21-51ef-b45a-683af7adbb8b


# **Part 2: Product Search**

In [ ]:
#@title API - Product Search
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

API_KEY = os.getenv("API_KEY")

# Function to fetch portfolio companies
def fetch_portfolio(input_port):
    url = f"https://api.securityscorecard.io/portfolios/{input_port}/companies"
    headers = {
        "accept": "application/json; charset=utf-8",
        "Authorization": f"Token {API_KEY}"
    }
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        json_data = response.json()
        output_df = pd.json_normalize(json_data.get("entries", []))
        output_df['Portfolio'] = input_port
        return output_df
    except Exception as e:
        print(f"Error fetching {input_port}: {e}")
        return pd.DataFrame()  # return empty DataFrame on failure

# List to hold results
df_vendors = pd.DataFrame()

# Use ThreadPoolExecutor for multithreading with progress bar
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(fetch_portfolio, port): port for port in input_ports}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching portfolios"):
        df_vendors = pd.concat([df_vendors, future.result()], ignore_index=True)

# Select specific fields
FIELDS = ["domain", "uuid", "name", "score", "added_date", "grade", "grade_url",
          "last30days_score_change", "industry", "size", "products", "products_count", "Portfolio"]
df_vendors = df_vendors[FIELDS]

# Save to Excel
df_vendors.to_excel("portfolio_vendors.xlsx", index=False)



Fetching portfolios: 100%|██████████| 1/1 [00:08<00:00,  8.82s/it]


In [ ]:
#@title Rename 'domain' to 'Vendor'
df_vendors.rename(columns={'domain': 'Vendor'}, inplace=True)

# Remove duplicate vendors, keeping the first occurrence
df_vendors = df_vendors.drop_duplicates(subset=['Vendor'], keep='first')

# Reset index
df_vendors = df_vendors.reset_index(drop=True)

# Save to CSV
df_vendors.to_csv("portfolio_vendors1.csv", index=False)


In [ ]:
#@title Product Matching
import pandas as pd
import ast
from tqdm import tqdm

# Define PRODUCT string (example: "react, next.js")
keywords = [k.strip().lower() for k in PRODUCT.split(',')]

# Read the CSV
df_vendors = pd.read_csv("portfolio_vendors1.csv")

# Function to safely parse products column
def parse_products(cell):
    try:
        if pd.isna(cell):
            return []
        elif isinstance(cell, list):
            return cell
        else:
            return ast.literal_eval(cell)
    except:
        return []

df_vendors['products'] = df_vendors['products'].apply(parse_products)

# List to collect results
results = []
total_products_scanned = 0

# Iterate over rows with progress bar
for _, row in tqdm(df_vendors.iterrows(), total=len(df_vendors), desc="Scanning vendors"):
    vendor = row['Vendor']
    products = row['products']

    if isinstance(products, list):
        total_products_scanned += len(products)
        for product in products:
            product_lower = product.lower()
            if any(keyword in product_lower for keyword in keywords):
                results.append({'Vendor': vendor, 'product_name': product})

print(f"Total products scanned: {total_products_scanned}")
print(f"Total matches found: {len(results)}")

# Create DataFrame of matches
df_matches = pd.DataFrame(results).reset_index(drop=True)
df_matches


Scanning vendors: 100%|██████████| 1239/1239 [00:00<00:00, 2642.14it/s]

Total products scanned: 577993
Total matches found: 979


,Vendor,product_name
0,ascendion.com,Microsoft SharePoint
1,ascendion.com,Microsoft SharePoint Online
2,penningtonslaw.com,Microsoft SharePoint
3,bv.com,Microsoft SharePoint
4,ao.com,Microsoft SharePoint
...,...,...
974,pultegroup.com,Microsoft Sharepoint Designer
975,cooperhealth.edu,Microsoft SharePoint
976,cooperhealth.edu,Microsoft SharePoint Online
977,greatamericaninsurancegroup.com,Microsoft SharePoint


In [ ]:
#@title Consolidated Section 2
#@title Input Dataframe
# Import libraries
import pandas as pd
import gspread
from google.colab import auth
import google.auth

# Authenticate with Google
auth.authenticate_user()
creds, _ = google.auth.default()
gc = gspread.authorize(creds)

# Open Google Sheet by URL
spreadsheet = gc.open_by_url("https://docs.google.com/spreadsheets/d/1dC25JdU4e9Ww1IhFr1vcZOgsuzFzXvdLDA251vfLKdE/edit?gid=0#gid=0")

# Select the first worksheet (or a specific one)
worksheet = spreadsheet.worksheet("Domain")

# Get all values (evaluated, not formulas)
data = worksheet.get_all_values()

# Get all values
data = worksheet.get_all_values()

# Set row 3 as header
header = data[0]

# Load data starting from row 4
df_section2 = pd.DataFrame(data[1:], columns=header)

#Remove the trail of Customer Count
df_section2['Suppliers'] = df_section2['Suppliers'].str.replace(r'Count = \d+\n', '', regex=True)

df_section2


,Vendor,Suppliers
0,leggmason.com,"ABBOTT LABS, Janus Henderson, Silphium Asset M..."
1,pimco.com,"ABBOTT LABS, Janus Henderson, Silphium Asset M..."
2,schroders.com,"ABBOTT LABS, Aegon, Janus Henderson, Lloyds, P..."
3,fidelityinternational.com,"ABBOTT LABS, Janus Henderson, Wilson Asset Man..."
4,point72.com,"ABBOTT LABS, Janus Henderson"
...,...,...
13111,zendata.security,ZENDATA Cybersecurity
13112,nttsecurity.com,ZENDATA Cybersecurity
13113,allenandovery.com,HCR Law
13114,orrick.com,HCR Law


# **Part 3: Processing**

In [ ]:
import pandas as pd

#@title Section 2 + Vendors with affected product

# Check if df_matches is empty and has no columns, which can lead to KeyError during merge
if df_matches.empty and len(df_matches.columns) == 0:
    # Create an empty DataFrame with the 'Vendor' column to avoid KeyError
    df_matches = pd.DataFrame(columns=['Vendor'])

df_merged = pd.merge(df_matches, df_section2, on='Vendor', how='inner')
df_merged

,Vendor,product_name,Suppliers
0,penningtonslaw.com,Microsoft SharePoint,"Bank of China, Irwin Mitchell"
1,ao.com,Microsoft SharePoint,Currys plc
2,jm.com,Microsoft SharePoint,National Gypsum Company
3,jm.com,Microsoft SharePoint Online,National Gypsum Company
4,thephoenixgroup.com,Microsoft SharePoint,"Aegon, Petrofac, Phoenix / ReAssure, PRA (Bank..."
...,...,...,...
733,intelerad.com,Microsoft SharePoint,"Carestream Health, Sectra, Telus Health"
734,2wglobal.com,Microsoft SharePoint,W&W Logistics
735,truist.com,Microsoft SharePoint,"City National Bank of Florida, Airstar Bank Li..."
736,truist.com,Microsoft Sharepoint Designer,"City National Bank of Florida, Airstar Bank Li..."


In [ ]:
import pandas as pd

# Step 1: Split Suppliers by comma and explode
df_exploded = df_merged.assign(
    Suppliers=df_merged['Suppliers'].str.split(',')
).explode('Suppliers')

# Step 2: Clean whitespace from supplier names
df_exploded['Suppliers'] = df_exploded['Suppliers'].str.strip()

# Step 3: Group by Supplier and product_name, aggregating unique vendors
df_grouped = df_exploded.groupby(['Suppliers', 'product_name'])['Vendor'].agg(lambda x: ', '.join(sorted(x.unique()))).reset_index()

# Step 4: Pivot so product_name becomes columns
df_pivot = df_grouped.pivot(index='Suppliers', columns='product_name', values='Vendor').reset_index()

# Step 5: Optionally rename Suppliers column to Customer
df_pivot.rename(columns={'Suppliers': 'Customer'}, inplace=True)

# Step 6: Sort by Customer alphabetically
df_pivot = df_pivot.sort_values('Customer').reset_index(drop=True)
df_pivot = df_pivot.fillna('')

df_pivot

product_name,Customer,Microsoft SharePoint,Microsoft SharePoint Online,Microsoft Sharepoint Designer,Microsoft Sharepoint Enterprise,Microsoft Sharepoint Foundation
0,ABBOTT LABS,"janushenderson.com, lazard.com, point72.com, s...","janushenderson.com, lazard.com, point72.com, s...",janushenderson.com,,
1,ABBYY Group,"abbyy.com, adobe.com, amazon.com, aon.com, atl...","abbyy.com, adobe.com, amazon.com, atlassian.co...","amazon.com, aon.com, att.com, cigna.com, fujit...","amazon.com, att.com, cigna.com, ey.com","att.com, ey.com"
2,ACCESSTAGE TECNOLOGIA SA,"accesstage.com.br, globalpayments.com, stripe.com",globalpayments.com,globalpayments.com,,
3,ACME,"adobe.com, adp.com, amazon.com, box.com, cbre....","adobe.com, adp.com, amazon.com, cbre.com, cisc...","adp.com, amazon.com, cisco.com, oracle.com, up...","amazon.com, cisco.com, eurofins.com","servicenow.com, ups.com, workday.com"
4,ANZ Bank,anz.com.au,,anz.com.au,,
...,...,...,...,...,...,...
625,dmg media,dmgmedia.co.uk,,,,
626,e92plus LTD,"crowdstrike.com, cyberark.com, paloaltonetwork...",cyberark.com,,,
627,iTOO Special Risks (Pty) Ltd,"allianz.com, chubb.com, libertymutual.com, zur...","allianz.com, chubb.com, libertymutual.com, zur...","libertymutual.com, zurich.com",,allianz.com
628,iso360,"aws.amazon.com, canva.com, dynatrace.com, newr...",dynatrace.com,,,


In [ ]:
#@title Customer Tracker
#@title Input Dataframe
# Import libraries
import pandas as pd
import gspread
from google.colab import auth
import google.auth

# Authenticate with Google
auth.authenticate_user()
creds, _ = google.auth.default()
gc = gspread.authorize(creds)

# Open Google Sheet by URL
spreadsheet = gc.open_by_url("https://docs.google.com/spreadsheets/d/1fCkhiefnp4rIl-VfQJSK1IxATbocOfam-fSQ1B3crug/edit?gid=870476161#gid=870476161")

# Select the first worksheet (or a specific one)
worksheet = spreadsheet.worksheet("Section2Tracker")

# Get all values (evaluated, not formulas)
data = worksheet.get_all_values()

# Set row 3 as header
header = data[1]

# Load data starting from row 4
df_customer_tracker = pd.DataFrame(data[2:], columns=header)

# Keep only the first 8 columns
df_customer_tracker = df_customer_tracker.iloc[:, :8]

# Replace empty strings with NaN
df_customer_tracker.iloc[:, 2:8] = df_customer_tracker.iloc[:, 2:8].replace('', pd.NA)

# Drop rows where all columns 3 to 8 are empty/NaN
df_customer_tracker = df_customer_tracker.dropna(
    subset=df_customer_tracker.columns[2:8],
    how='all'
).reset_index(drop=True)

df_customer_tracker['Customer'] = df_customer_tracker.pop('Company Name')

df_customer_tracker


,Reference Spreadsheet,Match in customer tracker,Status,Category,RGO,Lead Support,Domains,Customer
0,https://docs.google.com/spreadsheets/d/1Ozgxzb...,ABBOTT LABS,Frozen,3. Gartner 2023,Alex,Jude,abbott.com,ABBOTT LABS
1,https://docs.google.com/spreadsheets/d/1DNFj1B...,ABBYY Group,Renewal (was direct with SSC),0 - VIP Firm that wants Dashboard on Day -1 ea...,Lewis,Leslie,abbyy.com,ABBYY Group
2,https://docs.google.com/spreadsheets/d/1u8lppI...,Ace Consultants,Frozen,5. Project SCALE EMEA batch 2 (2024-09),Leslie,Keith,aceconsultants.fr,Ace Consultants
3,https://docs.google.com/spreadsheets/d/12vbNtn...,ACEN,Frozen,3. Prospect that is a current priority,Ivy,Keith,acenrenewables.com,ACEN
4,https://docs.google.com/spreadsheets/d/1fiQBCT...,ACME,<NA>,6. Low Priority,Ivy,Joan,acme.com,ACME
...,...,...,...,...,...,...,...,...
699,https://docs.google.com/spreadsheets/d/1SKbu6h...,The Collinson Group,Frozen,8b. Project 300 for Nadji - 2nd 50 priority - ...,Michael,<NA>,collinsongroup.com,The Collinson Group
700,https://docs.google.com/spreadsheets/d/1G_ymyV...,Lagaviti,New,4. Project Alumni,Leslie,Alex,lagaviti.is,Lagaviti
701,https://docs.google.com/spreadsheets/d/16rlz2W...,BEEAH Group,New,4. Project Alumni,Lewis,Alex,beeahgroup.com,BEEAH Group
702,https://docs.google.com/spreadsheets/d/1bTpFcz...,ZENDATA Cybersecurity,New,5 - BA Webinar's in 2024 - July 16th,Kobe,Alex,zendata.security,ZENDATA Cybersecurity


In [ ]:
#@title Customers + Vendors using Affected Vendors

df_final = pd.merge(df_pivot, df_customer_tracker, on='Customer', how='inner')
df_final

,Customer,Microsoft SharePoint,Microsoft SharePoint Online,Microsoft Sharepoint Designer,Microsoft Sharepoint Enterprise,Microsoft Sharepoint Foundation,Reference Spreadsheet,Match in customer tracker,Status,Category,RGO,Lead Support,Domains
0,ABBOTT LABS,"janushenderson.com, lazard.com, point72.com, s...","janushenderson.com, lazard.com, point72.com, s...",janushenderson.com,,,https://docs.google.com/spreadsheets/d/1Ozgxzb...,ABBOTT LABS,Frozen,3. Gartner 2023,Alex,Jude,abbott.com
1,ABBYY Group,"abbyy.com, adobe.com, amazon.com, aon.com, atl...","abbyy.com, adobe.com, amazon.com, atlassian.co...","amazon.com, aon.com, att.com, cigna.com, fujit...","amazon.com, att.com, cigna.com, ey.com","att.com, ey.com",https://docs.google.com/spreadsheets/d/1DNFj1B...,ABBYY Group,Renewal (was direct with SSC),0 - VIP Firm that wants Dashboard on Day -1 ea...,Lewis,Leslie,abbyy.com
2,ACME,"adobe.com, adp.com, amazon.com, box.com, cbre....","adobe.com, adp.com, amazon.com, cbre.com, cisc...","adp.com, amazon.com, cisco.com, oracle.com, up...","amazon.com, cisco.com, eurofins.com","servicenow.com, ups.com, workday.com",https://docs.google.com/spreadsheets/d/1fiQBCT...,ACME,<NA>,6. Low Priority,Ivy,Joan,acme.com
3,ANZ Bank,anz.com.au,,anz.com.au,,,https://docs.google.com/spreadsheets/d/18SQL8F...,ANZ Bank,New,6. Low Priority,Alex,Victor,anz.com.au
4,AO Foundation,"aofoundation.org, salesforce.com, stripe.com",,,,,https://docs.google.com/spreadsheets/d/1UWva7U...,AO Foundation,Renewal,5a. Project SCALE - EMEA - Jan 2024,Leslie,Ariadna,aofoundation.org
...,...,...,...,...,...,...,...,...,...,...,...,...,...
405,acQuire,"acquire.com.au, adobe.com, microsoft.com",adobe.com,,,,https://docs.google.com/spreadsheets/d/13yzWcP...,acQuire,Renewal,6a. Project SCALE - APAC - Jun 2024,Ariadna,Michael,acquire.com.au
406,ctm Information Technology,cbre.com,cbre.com,,,,https://docs.google.com/spreadsheets/d/1f7Tcap...,ctm Information Technology,Frozen,3. InfoSec - June 2024,Alex,<NA>,ctm.com
407,dmg media,dmgmedia.co.uk,,,,,https://docs.google.com/spreadsheets/d/1dJU889...,dmg media,Renewal,1. VIP Firm - MAX Prospect 2025,Alex,Leslie,dmgmedia.co.uk
408,e92plus LTD,"crowdstrike.com, cyberark.com, paloaltonetwork...",cyberark.com,,,,https://docs.google.com/spreadsheets/d/11sO06x...,e92plus Ltd,Frozen,3. Project Phoenix,Nico,<NA>,https://www.e92plus.com/


# **Excel Generation**

In [ ]:
#@title Excel Generation
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# Assume df_final is your DataFrame

# Strip extra spaces from Status
df_final['Status'] = df_final['Status'].astype(str).str.strip()

# Split into Frozen and Active
df_frozen = df_final[df_final['Status'].str.lower() == 'frozen'].copy()
df_active = df_final[df_final['Status'].str.lower() != 'frozen'].copy()

# Sort by Category alphabetically
df_frozen = df_frozen.sort_values('Category').reset_index(drop=True)
df_active = df_active.sort_values('Category').reset_index(drop=True)

# Output to Excel with two sheets
excel_path = "Customer_Tracker.xlsx"
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_active.to_excel(writer, sheet_name='Active Firms', index=False)
    df_frozen.to_excel(writer, sheet_name='Frozen Firms', index=False)

# Open workbook to format
wb = load_workbook(excel_path)

for sheet_name in ['Active Firms', 'Frozen Firms']:
    ws = wb[sheet_name]

    # Header formatting
    header_fill = PatternFill(start_color="FFC000", end_color="FFC000", fill_type="solid")  # orange header
    header_font = Font(name='Arial', size=10, bold=True)
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center', vertical='center')

    # Auto-fit columns with min 100px (~13.3 chars) and max 300px (~40 chars)
    for col in ws.columns:
        max_length = 0
        column = col[0].column_letter
        for cell in col:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        # Convert max_length to approximate pixels
        width = min(max(max_length + 2, 13.3), 40)  # 13.3 chars ~ 100px, 40 chars ~ 300px
        ws.column_dimensions[column].width = width

wb.save(excel_path)


In [ ]:
from google.colab import drive
import shutil
from datetime import datetime
import os

# Set your product name
PRODUCT = "Reactjs, next.js"  # replace with your actual product

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Local file path
local_file = '/content/Customer_Tracker.xlsx'

# Get current date
today = datetime.now().strftime('%Y-%m-%d')

# Destination folder path
dest_folder = '/content/drive/Shareddrives/Cyber Rescue Team/SCCC - Supply Chain Cyber Crisis Automation/SCCC-Result'

# Create folder if it doesn't exist
os.makedirs(dest_folder, exist_ok=True)

# Destination file path
drive_path = os.path.join(dest_folder, f'SCCC Result from All CR Customer - {PRODUCT} - {today}.xlsx')

# Copy and rename the file
shutil.copy(local_file, drive_path)

print(f'File saved to Google Drive as: {drive_path}')


Mounted at /content/drive
File saved to Google Drive as: /content/drive/Shareddrives/Cyber Rescue Team/SCCC - Supply Chain Cyber Crisis Automation/SCCC-Result/SCCC Result from All CR Customer - Reactjs, next.js - 2026-08-07.xlsx
